# 🛠️ GenAI Course — Module 4: Setting Up Your MCP Environment

In this module we go from zero to a **running MCP server** connected to Claude. By the end you will have:
- ✅ All prerequisites installed and verified
- ✅ The MCP SDK set up in both Python and TypeScript
- ✅ Launched and debugged your first MCP server with the MCP Inspector
- ✅ Connected that server to Claude Desktop

---

### 📚 Topics Covered
1. Prerequisites — Node.js, Python, package managers
2. Installing the MCP SDK (TypeScript / Python)
3. MCP Inspector Tool — testing and debugging servers
4. Running Your First MCP Server Locally
5. Connecting MCP Servers to Claude Desktop / Claude.ai

---
> 💡 **How to use this notebook:** Markdown cells contain theory and diagrams. Code cells marked `[SHELL]` show terminal commands — run them in your terminal, not inside Jupyter. Code cells marked `[PYTHON]` can be executed directly here.

---
## 1. Prerequisites — Node.js, Python & Package Managers

MCP servers can be written in **Python** or **TypeScript (Node.js)**. The table below lists what you need depending on your language of choice:

| Requirement | Minimum Version | Check Command | Purpose |
|-------------|----------------|---------------|---------|
| Node.js | 18+ (LTS) | `node --version` | Run TypeScript MCP servers |
| npm | 9+ | `npm --version` | Install TS SDK & tools |
| Python | 3.10+ | `python3 --version` | Run Python MCP servers |
| pip | 22+ | `pip --version` | Install Python SDK |
| uv *(recommended)* | 0.4+ | `uv --version` | Fast Python pkg manager from Astral |
| Claude Desktop | latest | — | MCP host for local testing |

### Why `uv`?
`uv` is a modern, Rust-based Python package manager that creates virtual environments and installs packages **10–100× faster** than pip. The official MCP quickstart uses `uv` for Python servers.

In [1]:
# [PYTHON] ── Check what's already installed ───────────────────────────────────
import subprocess, sys

def check_tool(name: str, args: list[str]) -> str:
    try:
        result = subprocess.run(args, capture_output=True, text=True, timeout=5)
        version = result.stdout.strip() or result.stderr.strip()
        return f"✅  {name:15s} {version.splitlines()[0]}"
    except FileNotFoundError:
        return f"❌  {name:15s} NOT FOUND — install required"
    except Exception as e:
        return f"⚠️  {name:15s} error: {e}"

checks = [
    ("Python",  ["python3", "--version"]),
    ("pip",     ["pip", "--version"]),
    ("uv",      ["uv", "--version"]),
    ("Node.js", ["node", "--version"]),
    ("npm",     ["npm", "--version"]),
    ("npx",     ["npx", "--version"]),
]

print("── Environment Check ──────────────────────────")
for name, args in checks:
    print(check_tool(name, args))
print("───────────────────────────────────────────────")

── Environment Check ──────────────────────────
✅  Python          Python was not found; run without arguments to install from the Microsoft Store, or disable this shortcut from Settings > Apps > Advanced app settings > App execution aliases.
✅  pip             pip 24.2 from C:\Users\LotusBlue\anaconda3\Lib\site-packages\pip (python 3.12)
✅  uv              uv 0.8.0 (0b2357294 2025-07-17)
✅  Node.js         v22.12.0
❌  npm             NOT FOUND — install required
❌  npx             NOT FOUND — install required
───────────────────────────────────────────────


In [3]:
# [SHELL commands — run in your terminal]
# ══════════════════════════════════════════════════════════════════
# Installing Node.js via nvm (recommended, works on macOS/Linux)
# ══════════════════════════════════════════════════════════════════

install_instructions = """
# ── Install nvm (Node Version Manager) ──────────────────────────
curl -o- https://raw.githubusercontent.com/nvm-sh/nvm/v0.39.7/install.sh | bash
source ~/.bashrc   # or ~/.zshrc on macOS

# ── Install Node.js LTS ──────────────────────────────────────────
nvm install --lts
nvm use --lts
node --version     # should print v20.x.x or v22.x.x

# ── Install uv (Python fast package manager) ────────────────────
curl -LsSf https://astral.sh/uv/install.sh | sh
source ~/.bashrc
uv --version       # should print uv 0.4.x or later

# ── Windows (PowerShell equivalents) ────────────────────────────
# Node: download installer from https://nodejs.org
# uv:   powershell -c "irm https://astral.sh/uv/install.ps1 | iex"
"""

print(install_instructions)


# ── Install nvm (Node Version Manager) ──────────────────────────
curl -o- https://raw.githubusercontent.com/nvm-sh/nvm/v0.39.7/install.sh | bash
source ~/.bashrc   # or ~/.zshrc on macOS

# ── Install Node.js LTS ──────────────────────────────────────────
nvm install --lts
nvm use --lts
node --version     # should print v20.x.x or v22.x.x

# ── Install uv (Python fast package manager) ────────────────────
curl -LsSf https://astral.sh/uv/install.sh | sh
source ~/.bashrc
uv --version       # should print uv 0.4.x or later

# ── Windows (PowerShell equivalents) ────────────────────────────
# Node: download installer from https://nodejs.org
# uv:   powershell -c "irm https://astral.sh/uv/install.ps1 | iex"



---
## 2. Installing the MCP SDK

Anthropic publishes official SDKs for both **Python** and **TypeScript**.

| SDK | Package | Install command |
|-----|---------|----------------|
| Python | `mcp` | `pip install mcp` or `uv add mcp` |
| TypeScript | `@modelcontextprotocol/sdk` | `npm install @modelcontextprotocol/sdk` |

Both SDKs provide:
- **Server classes** — build and expose tools, resources, prompts
- **Client classes** — connect to and call remote MCP servers
- **Transport layers** — stdio (local) and HTTP (remote)
- **Type definitions** — full TypeScript types / Python dataclasses

In [5]:
# [SHELL] ── Python SDK Installation ──────────────────────────────────────────

python_sdk_setup = """
# Option A — using pip (simple projects)
pip install mcp

# Option B — using uv in a new project (recommended)
mkdir my-mcp-server
cd my-mcp-server
uv init                  # creates pyproject.toml
uv add mcp               # adds mcp to dependencies
uv venv                  # create virtual environment in .venv/
source .venv/bin/activate

# Verify
python3 -c "import mcp; print('mcp version:', mcp.__version__)"
"""
print("Python SDK setup:\n", python_sdk_setup)

Python SDK setup:
 
# Option A — using pip (simple projects)
pip install mcp

# Option B — using uv in a new project (recommended)
mkdir my-mcp-server
cd my-mcp-server
uv init                  # creates pyproject.toml
uv add mcp               # adds mcp to dependencies
uv venv                  # create virtual environment in .venv/
source .venv/bin/activate

# Verify
python3 -c "import mcp; print('mcp version:', mcp.__version__)"



In [7]:
# [SHELL] ── TypeScript SDK Installation ──────────────────────────────────────

ts_sdk_setup = """
# 1. Create a new project
mkdir my-ts-mcp-server
cd my-ts-mcp-server
npm init -y

# 2. Install MCP SDK + TypeScript tooling
npm install @modelcontextprotocol/sdk zod
npm install -D @types/node typescript

# 3. Add a tsconfig.json
npx tsc --init --target ES2022 --module Node16 --moduleResolution Node16 --outDir build

# 4. Update package.json — add these fields:
#   "type": "module",
#   "scripts": { "build": "tsc", "start": "node build/index.js" }

# Verify
node -e "const { McpServer } = require('@modelcontextprotocol/sdk/server/mcp.js'); console.log('SDK loaded!')"
"""
print("TypeScript SDK setup:\n", ts_sdk_setup)

TypeScript SDK setup:
 
# 1. Create a new project
mkdir my-ts-mcp-server
cd my-ts-mcp-server
npm init -y

# 2. Install MCP SDK + TypeScript tooling
npm install @modelcontextprotocol/sdk zod
npm install -D @types/node typescript

# 3. Add a tsconfig.json
npx tsc --init --target ES2022 --module Node16 --moduleResolution Node16 --outDir build

# 4. Update package.json — add these fields:
#   "type": "module",
#   "scripts": { "build": "tsc", "start": "node build/index.js" }

# Verify
node -e "const { McpServer } = require('@modelcontextprotocol/sdk/server/mcp.js'); console.log('SDK loaded!')"



In [9]:
# [PYTHON] ── Verify Python SDK is importable (run after pip install mcp) ─────

try:
    import mcp
    from mcp.server import Server
    from mcp.server.stdio import stdio_server
    from mcp import types
    print(f"✅ Python MCP SDK imported successfully")
    print(f"   Key classes available: Server, stdio_server, types")

    # Show available tool/resource types
    public_api = [x for x in dir(types) if not x.startswith("_")]
    print(f"   mcp.types exports: {public_api[:10]} ... ({len(public_api)} total)")

except ImportError:
    print("❌ mcp not installed. Run: pip install mcp")

✅ Python MCP SDK imported successfully
   Key classes available: Server, stdio_server, types
   mcp.types exports: ['Annotated', 'Annotations', 'Any', 'AnyFunction', 'AnyUrl', 'AudioContent', 'BaseMetadata', 'BaseModel', 'BlobResourceContents', 'CONNECTION_CLOSED'] ... (246 total)


In [11]:
# [PYTHON] ── Explore the Python MCP SDK structure ─────────────────────────────

try:
    from mcp.server import Server
    import inspect

    print("Server class public methods:")
    methods = [
        (name, m)
        for name, m in inspect.getmembers(Server, predicate=inspect.isfunction)
        if not name.startswith("_")
    ]
    for name, m in methods[:12]:
        sig = str(inspect.signature(m))[:60]
        print(f"  • {name}{sig}")
except ImportError:
    print("Run 'pip install mcp' first, then re-run this cell.")

Server class public methods:
  • call_tool(self, *, validate_input: 'bool' = True)
  • completion(self)
  • create_initialization_options(self, notification_options: 'NotificationOptions | None' = 
  • get_capabilities(self, notification_options: 'NotificationOptions', experime
  • get_prompt(self)
  • list_prompts(self)
  • list_resource_templates(self)
  • list_resources(self)
  • list_tools(self)
  • progress_notification(self)
  • read_resource(self)
  • run(self, read_stream: 'MemoryObjectReceiveStream[SessionMessag


---
## 3. MCP Inspector — Testing and Debugging Servers

The **MCP Inspector** is an official browser-based dev tool for interacting with any MCP server without needing Claude Desktop.

```
Your Terminal
┌──────────────────────────────────────────────┐
│  npx @modelcontextprotocol/inspector server.py│
│                                              │
│  MCP Inspector running at http://localhost:5173 │
└──────────────────────────────────────────────┘
           │
           ▼
┌─────────────────────────────┐
│  Browser — Inspector UI     │
│  • List tools               │
│  • Call tools with params   │
│  • Browse resources         │
│  • View raw JSON-RPC msgs   │
└─────────────────────────────┘
```

### Inspector Features
| Tab | What you can do |
|-----|-----------------|
| **Tools** | See all tools, click to call them with a form |
| **Resources** | Browse and read resources exposed by the server |
| **Prompts** | List and preview prompt templates |
| **Messages** | Full JSON-RPC 2.0 message log — great for debugging |

In [13]:
# [SHELL] ── Launching the MCP Inspector ──────────────────────────────────────

inspector_commands = """
# ── Against a Python server ─────────────────────────────────────────
npx @modelcontextprotocol/inspector python3 server.py

# ── Against a TypeScript (compiled) server ──────────────────────────
npx @modelcontextprotocol/inspector node build/index.js

# ── Against a uv-managed server ─────────────────────────────────────
npx @modelcontextprotocol/inspector uv run server.py

# ── Pass environment variables ───────────────────────────────────────
MCP_ENV=dev npx @modelcontextprotocol/inspector python3 server.py

# Inspector opens at: http://localhost:5173
# Press Ctrl+C to stop it.
"""
print(inspector_commands)


# ── Against a Python server ─────────────────────────────────────────
npx @modelcontextprotocol/inspector python3 server.py

# ── Against a TypeScript (compiled) server ──────────────────────────
npx @modelcontextprotocol/inspector node build/index.js

# ── Against a uv-managed server ─────────────────────────────────────
npx @modelcontextprotocol/inspector uv run server.py

# ── Pass environment variables ───────────────────────────────────────
MCP_ENV=dev npx @modelcontextprotocol/inspector python3 server.py

# Inspector opens at: http://localhost:5173
# Press Ctrl+C to stop it.



In [15]:
# [PYTHON] ── Simulate what the Inspector does under the hood ─────────────────
# The Inspector is just an MCP client that sends standard JSON-RPC messages.
# Here we replicate the first two calls it makes:

import json

def inspector_init_sequence():
    """Return the ordered JSON-RPC messages the Inspector sends on startup."""
    return [
        {
            "step": 1,
            "direction": "Inspector → Server",
            "message": {
                "jsonrpc": "2.0", "id": 1, "method": "initialize",
                "params": {
                    "protocolVersion": "2024-11-05",
                    "capabilities": {"roots": {"listChanged": True}, "sampling": {}},
                    "clientInfo": {"name": "MCP Inspector", "version": "0.8.0"}
                }
            }
        },
        {
            "step": 2,
            "direction": "Inspector → Server",
            "message": {"jsonrpc": "2.0", "method": "notifications/initialized"}
        },
        {
            "step": 3,
            "direction": "Inspector → Server  (populates Tools tab)",
            "message": {"jsonrpc": "2.0", "id": 2, "method": "tools/list", "params": {}}
        },
        {
            "step": 4,
            "direction": "Inspector → Server  (populates Resources tab)",
            "message": {"jsonrpc": "2.0", "id": 3, "method": "resources/list", "params": {}}
        },
    ]

print("📡 Messages MCP Inspector sends at startup:\n")
for step in inspector_init_sequence():
    print(f"Step {step['step']}  [{step['direction']}]")
    print(json.dumps(step["message"], indent=4))
    print()

📡 Messages MCP Inspector sends at startup:

Step 1  [Inspector → Server]
{
    "jsonrpc": "2.0",
    "id": 1,
    "method": "initialize",
    "params": {
        "protocolVersion": "2024-11-05",
        "capabilities": {
            "roots": {
                "listChanged": true
            },
            "sampling": {}
        },
        "clientInfo": {
            "name": "MCP Inspector",
            "version": "0.8.0"
        }
    }
}

Step 2  [Inspector → Server]
{
    "jsonrpc": "2.0",
    "method": "notifications/initialized"
}

Step 3  [Inspector → Server  (populates Tools tab)]
{
    "jsonrpc": "2.0",
    "id": 2,
    "method": "tools/list",
    "params": {}
}

Step 4  [Inspector → Server  (populates Resources tab)]
{
    "jsonrpc": "2.0",
    "id": 3,
    "method": "resources/list",
    "params": {}
}



---
## 4. Running Your First MCP Server Locally

We'll build and run a **Calculator MCP server** in both Python and TypeScript.

### Architecture
```
┌───────────────┐  stdio (stdin/stdout)  ┌──────────────────────┐
│  MCP Client   │ ◄───────────────────► │   MCP Server Process  │
│ (Inspector /  │   JSON-RPC 2.0 msgs   │   server.py / index.js│
│  Claude Desk) │                        │   • tool: add         │
└───────────────┘                        │   • tool: multiply    │
                                         │   • tool: history     │
                                         └──────────────────────┘
```

> ⚠️ **STDIO logging rule:** For stdio-based servers, **never** write to stdout (`print()` in Python, `console.log()` in JS). It corrupts the JSON-RPC stream. Always use `sys.stderr` / `console.error()` for debug output.

In [17]:
# [PYTHON] ── server.py — Full Calculator MCP Server ──────────────────────────
# Save this cell's content to a file named 'server.py' and run it.

server_py = '''
#!/usr/bin/env python3
"""
calculator_server.py  —  A minimal MCP server exposing math tools.
Run with:  python3 calculator_server.py
Debug with: npx @modelcontextprotocol/inspector python3 calculator_server.py
"""

import asyncio
import sys
from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp import types

# ── Create the server instance ────────────────────────────────────────────────
app = Server("calculator-server")

# In-memory operation history
_history: list[str] = []

# ── Register tools ────────────────────────────────────────────────────────────
@app.list_tools()
async def list_tools() -> list[types.Tool]:
    return [
        types.Tool(
            name="add",
            description="Add two numbers together.",
            inputSchema={
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "First operand"},
                    "b": {"type": "number", "description": "Second operand"}
                },
                "required": ["a", "b"]
            }
        ),
        types.Tool(
            name="multiply",
            description="Multiply two numbers together.",
            inputSchema={
                "type": "object",
                "properties": {
                    "a": {"type": "number"},
                    "b": {"type": "number"}
                },
                "required": ["a", "b"]
            }
        ),
        types.Tool(
            name="history",
            description="Return all operations performed this session.",
            inputSchema={"type": "object", "properties": {}}
        ),
    ]

@app.call_tool()
async def call_tool(name: str, arguments: dict) -> list[types.TextContent]:
    print(f"[server] tool called: {name} args={arguments}", file=sys.stderr)  # ✅ stderr only!

    if name == "add":
        result = arguments["a"] + arguments["b"]
        expr = f"{arguments[\'a\']} + {arguments[\'b\']} = {result}"
        _history.append(expr)
        return [types.TextContent(type="text", text=str(result))]

    elif name == "multiply":
        result = arguments["a"] * arguments["b"]
        expr = f"{arguments[\'a\']} × {arguments[\'b\']} = {result}"
        _history.append(expr)
        return [types.TextContent(type="text", text=str(result))]

    elif name == "history":
        if not _history:
            return [types.TextContent(type="text", text="No operations yet.")]
        return [types.TextContent(type="text", text="\\n".join(f"{i+1}. {op}" for i, op in enumerate(_history)))]

    else:
        raise ValueError(f"Unknown tool: {name}")

# ── Main entrypoint ───────────────────────────────────────────────────────────
async def main():
    print("[server] Calculator MCP server starting on stdio...", file=sys.stderr)
    async with stdio_server() as streams:
        await app.run(streams[0], streams[1], app.create_initialization_options())

if __name__ == "__main__":
    asyncio.run(main())
'''

# Write the server file
with open("calculator_server.py", "w") as f:
    f.write(server_py)

print("✅ calculator_server.py written.")
print("\nTo test it, run in your terminal:")
print("  npx @modelcontextprotocol/inspector python3 calculator_server.py")
print("Then open  http://localhost:5173  in your browser.")

UnicodeEncodeError: 'charmap' codec can't encode characters in position 354-355: character maps to <undefined>

In [19]:
# [PYTHON] ── TypeScript equivalent — index.ts ─────────────────────────────────
# For students choosing TypeScript

ts_server = '''
// index.ts  —  Calculator MCP Server (TypeScript)
// Build:   npm run build
// Inspect: npx @modelcontextprotocol/inspector node build/index.js

import { McpServer } from "@modelcontextprotocol/sdk/server/mcp.js";
import { StdioServerTransport } from "@modelcontextprotocol/sdk/server/stdio.js";
import { z } from "zod";

const server = new McpServer({
  name: "calculator-server",
  version: "1.0.0",
});

const history: string[] = [];

// ── Register tools ────────────────────────────────────────────────────────────
server.tool(
  "add",
  "Add two numbers together.",
  { a: z.number().describe("First operand"), b: z.number().describe("Second operand") },
  async ({ a, b }) => {
    const result = a + b;
    history.push(`${a} + ${b} = ${result}`);
    return { content: [{ type: "text", text: String(result) }] };
  }
);

server.tool(
  "multiply",
  "Multiply two numbers together.",
  { a: z.number(), b: z.number() },
  async ({ a, b }) => {
    const result = a * b;
    history.push(`${a} × ${b} = ${result}`);
    return { content: [{ type: "text", text: String(result) }] };
  }
);

server.tool(
  "history",
  "Return all operations performed this session.",
  {},
  async () => {
    const text = history.length
      ? history.map((op, i) => `${i + 1}. ${op}`).join("\\n")
      : "No operations yet.";
    return { content: [{ type: "text", text }] };
  }
);

// ── Start server ──────────────────────────────────────────────────────────────
const transport = new StdioServerTransport();
await server.connect(transport);
// ✅ Never use console.log() here — it corrupts the stdio JSON-RPC stream!
console.error("Calculator MCP server running on stdio");
'''

with open("index.ts", "w") as f:
    f.write(ts_server)

print("✅ index.ts written.")
print("\nTo compile and run:")
print("  npm run build")
print("  npx @modelcontextprotocol/inspector node build/index.js")

UnicodeEncodeError: 'charmap' codec can't encode characters in position 459-460: character maps to <undefined>

In [21]:
# [PYTHON] ── Simulate server startup & tool execution (no external process) ───
# This lets you understand the server logic before running the real process.

class SimulatedCalcServer:
    """Mirrors the logic in calculator_server.py without needing the MCP SDK."""

    def __init__(self):
        self._history = []
        self.name = "calculator-server"

    def list_tools(self):
        return ["add", "multiply", "history"]

    def call_tool(self, name: str, arguments: dict):
        if name == "add":
            result = arguments["a"] + arguments["b"]
            self._history.append(f"{arguments['a']} + {arguments['b']} = {result}")
            return {"content": [{"type": "text", "text": str(result)}], "isError": False}
        elif name == "multiply":
            result = arguments["a"] * arguments["b"]
            self._history.append(f"{arguments['a']} × {arguments['b']} = {result}")
            return {"content": [{"type": "text", "text": str(result)}], "isError": False}
        elif name == "history":
            text = "\n".join(f"{i+1}. {op}" for i, op in enumerate(self._history)) or "No operations yet."
            return {"content": [{"type": "text", "text": text}], "isError": False}
        else:
            return {"content": [{"type": "text", "text": f"Unknown tool: {name}"}], "isError": True}

# ── Run the simulation ────────────────────────────────────────────────────────
sim = SimulatedCalcServer()

print(f"Server name: {sim.name}")
print(f"Tools: {sim.list_tools()}\n")

operations = [
    ("add",      {"a": 25, "b": 17}),
    ("multiply", {"a": 6,  "b": 7}),
    ("add",      {"a": 100, "b": -45}),
    ("history",  {}),
]

for tool, args in operations:
    result = sim.call_tool(tool, args)
    output = result["content"][0]["text"]
    status = "✅" if not result["isError"] else "❌"
    print(f"{status} {tool}({', '.join(f'{k}={v}' for k,v in args.items())})")
    print(f"   → {output}\n")

Server name: calculator-server
Tools: ['add', 'multiply', 'history']

✅ add(a=25, b=17)
   → 42

✅ multiply(a=6, b=7)
   → 42

✅ add(a=100, b=-45)
   → 55

✅ history()
   → 1. 25 + 17 = 42
2. 6 × 7 = 42
3. 100 + -45 = 55



---
## 5. Connecting MCP Servers to Claude Desktop / Claude.ai

### How Claude Desktop manages servers
Claude Desktop reads a single JSON config file at startup. For each entry it **spawns a subprocess**, communicates via stdio, and makes all exposed tools available in the chat.

```
Claude Desktop startup
       │
       ▼
Read  claude_desktop_config.json
       │
       ├─ For each mcpServer entry:
       │    spawn process (command + args + env)
       │    run MCP initialize handshake
       │    register tools in UI (🔨 hammer icon)
       │
       ▼
Ready — tools appear in chat
```

### Config file locations

| OS | Path |
|----|------|
| macOS | `~/Library/Application Support/Claude/claude_desktop_config.json` |
| Windows | `%APPDATA%\Claude\claude_desktop_config.json` |
| Linux | *(Claude Desktop not yet available; use MCP Inspector or Claude API)* |

> 💡 Shortcut: in Claude Desktop go to **Settings → Developer → Edit Config**

In [25]:
# [PYTHON] ── Explore & generate claude_desktop_config.json ────────────────────

import json, os, pathlib

def get_config_path() -> pathlib.Path | None:
    """Return the platform-specific config file path."""
    if os.name == "posix":
        if "darwin" in os.uname().sysname.lower():
            return pathlib.Path.home() / "Library/Application Support/Claude/claude_desktop_config.json"
        return None  # Linux
    elif os.name == "nt":
        return pathlib.Path(os.environ["APPDATA"]) / "Claude/claude_desktop_config.json"
    return None

config_path = get_config_path()
if config_path:
    print(f"Config file location on your system:\n  {config_path}")
    print(f"  Exists: {config_path.exists()}")
else:
    print("Linux detected — Claude Desktop not available. Use MCP Inspector or the API.")

Config file location on your system:
  C:\Users\LotusBlue\AppData\Roaming\Claude\claude_desktop_config.json
  Exists: False


In [1]:
# [PYTHON] ── Config templates for different server types ─────────────────────

import json

configs = {
    "Python server (direct)": {
        "mcpServers": {
            "calculator": {
                "command": "python3",
                "args": ["/ABSOLUTE/PATH/TO/calculator_server.py"]
                # ⚠️  Always use ABSOLUTE paths — relative paths don't work!
            }
        }
    },
    "Python server (uv venv)": {
        "mcpServers": {
            "calculator": {
                "command": "/ABSOLUTE/PATH/TO/project/.venv/bin/python",
                "args": ["/ABSOLUTE/PATH/TO/project/calculator_server.py"]
                # Use the venv python so all dependencies (mcp, etc.) are available
            }
        }
    },
    "TypeScript server (node)": {
        "mcpServers": {
            "calculator": {
                "command": "node",
                "args": ["/ABSOLUTE/PATH/TO/build/index.js"]
            }
        }
    },
    "Multiple servers": {
        "mcpServers": {
            "calculator": {
                "command": "python3",
                "args": ["/path/to/calculator_server.py"]
            },
            "filesystem": {
                "command": "npx",
                "args": ["-y", "@modelcontextprotocol/server-filesystem", "/Users/username/Desktop"]
            },
            "brave-search": {
                "command": "npx",
                "args": ["-y", "@modelcontextprotocol/server-brave-search"],
                "env": {
                    "BRAVE_API_KEY": "YOUR_BRAVE_API_KEY_HERE"
                    # ⚠️ Secrets go in 'env', never hardcoded in args
                }
            }
        }
    }
}

for label, cfg in configs.items():
    print(f"\n{'═'*55}")
    print(f"  {label}")
    print('═'*55)
    print(json.dumps(cfg, indent=2))


═══════════════════════════════════════════════════════
  Python server (direct)
═══════════════════════════════════════════════════════
{
  "mcpServers": {
    "calculator": {
      "command": "python3",
      "args": [
        "/ABSOLUTE/PATH/TO/calculator_server.py"
      ]
    }
  }
}

═══════════════════════════════════════════════════════
  Python server (uv venv)
═══════════════════════════════════════════════════════
{
  "mcpServers": {
    "calculator": {
      "command": "/ABSOLUTE/PATH/TO/project/.venv/bin/python",
      "args": [
        "/ABSOLUTE/PATH/TO/project/calculator_server.py"
      ]
    }
  }
}

═══════════════════════════════════════════════════════
  TypeScript server (node)
═══════════════════════════════════════════════════════
{
  "mcpServers": {
    "calculator": {
      "command": "node",
      "args": [
        "/ABSOLUTE/PATH/TO/build/index.js"
      ]
    }
  }
}

═══════════════════════════════════════════════════════
  Multiple servers
══════════════

In [3]:
# [PYTHON] ── Config helper: generate a config entry for the calculator server ─

import pathlib, json, sys

def generate_config_entry(server_name: str, script_path: str, use_venv: bool = False):
    """
    Generate a claude_desktop_config.json entry.
    
    Parameters
    ----------
    server_name : str   — key used in the mcpServers object
    script_path : str   — absolute path to server.py
    use_venv    : bool  — whether to use .venv/bin/python instead of system python
    """
    script = pathlib.Path(script_path).resolve()
    if use_venv:
        python_bin = script.parent / ".venv" / "bin" / "python"
    else:
        python_bin = pathlib.Path(sys.executable)

    entry = {
        "mcpServers": {
            server_name: {
                "command": str(python_bin),
                "args": [str(script)]
            }
        }
    }
    return entry

# Generate a config for the calculator server we created earlier
script_path = pathlib.Path("calculator_server.py").resolve()
config = generate_config_entry("calculator", str(script_path))

print("Generated claude_desktop_config.json entry:")
print(json.dumps(config, indent=2))
print()
print("Steps after saving this config:")
print("  1. Copy the JSON above into your claude_desktop_config.json file")
print("  2. Quit Claude Desktop completely (Cmd+Q / Alt+F4)")
print("  3. Relaunch Claude Desktop")
print("  4. Look for the 🔨 hammer icon in the chat input bar")
print("  5. Click it — you should see 'add', 'multiply', 'history'")

Generated claude_desktop_config.json entry:
{
  "mcpServers": {
    "calculator": {
      "command": "C:\\Users\\LotusBlue\\anaconda3\\python.exe",
      "args": [
        "C:\\Users\\LotusBlue\\Coding\\GenAI\\Course\\calculator_server.py"
      ]
    }
  }
}

Steps after saving this config:
  1. Copy the JSON above into your claude_desktop_config.json file
  2. Quit Claude Desktop completely (Cmd+Q / Alt+F4)
  3. Relaunch Claude Desktop
  4. Look for the 🔨 hammer icon in the chat input bar
  5. Click it — you should see 'add', 'multiply', 'history'


In [5]:
# [PYTHON] ── Troubleshooting common connection problems ───────────────────────

troubleshooting = [
    {
        "symptom":  "🔨 hammer icon not appearing after restart",
        "causes":   ["Config file has a JSON syntax error", "Wrong file path", "Claude not fully restarted"],
        "fixes":    [
            "Validate JSON: python3 -c \"import json; json.load(open('claude_desktop_config.json'))\"",
            "Use absolute paths only (no ~ or relative paths)",
            "Quit via Cmd+Q, wait 3s, relaunch"
        ]
    },
    {
        "symptom":  "⚠️  Server shown as 'failed' in Claude settings",
        "causes":   ["Server script crashes on startup", "Missing dependency", "Wrong python interpreter"],
        "fixes":    [
            "Run the server directly: python3 calculator_server.py  — look for errors",
            "Activate your venv and check: pip list | grep mcp",
            "Point 'command' to the venv python: /path/to/.venv/bin/python"
        ]
    },
    {
        "symptom":  "🔇 Server starts but tools are missing",
        "causes":   ["list_tools() handler not registered", "Server crashed after init"],
        "fixes":    [
            "Confirm @app.list_tools() decorator is applied",
            "Test with Inspector: npx @modelcontextprotocol/inspector python3 server.py"
        ]
    },
    {
        "symptom":  "💥 Corrupted JSON-RPC stream / garbled output",
        "causes":   ["print() used in server instead of sys.stderr"],
        "fixes":    [
            "Replace all print() with print(..., file=sys.stderr)",
            "In TypeScript, replace console.log() with console.error()"
        ]
    },
]

print("═" * 60)
print("  MCP Server Troubleshooting Guide")
print("═" * 60)

for item in troubleshooting:
    print(f"\n{item['symptom']}")
    print("  Possible causes:")
    for c in item["causes"]:
        print(f"    - {c}")
    print("  Fixes:")
    for f in item["fixes"]:
        print(f"    → {f}")

════════════════════════════════════════════════════════════
  MCP Server Troubleshooting Guide
════════════════════════════════════════════════════════════

🔨 hammer icon not appearing after restart
  Possible causes:
    - Config file has a JSON syntax error
    - Wrong file path
    - Claude not fully restarted
  Fixes:
    → Validate JSON: python3 -c "import json; json.load(open('claude_desktop_config.json'))"
    → Use absolute paths only (no ~ or relative paths)
    → Quit via Cmd+Q, wait 3s, relaunch

⚠️  Server shown as 'failed' in Claude settings
  Possible causes:
    - Server script crashes on startup
    - Missing dependency
    - Wrong python interpreter
  Fixes:
    → Run the server directly: python3 calculator_server.py  — look for errors
    → Activate your venv and check: pip list | grep mcp
    → Point 'command' to the venv python: /path/to/.venv/bin/python

🔇 Server starts but tools are missing
  Possible causes:
    - list_tools() handler not registered
    - Server

In [7]:
# [PYTHON] ── Using pre-built Anthropic reference servers ─────────────────────
# Anthropic ships 6 production-ready servers you can add with one npx line:

reference_servers = [
    {
        "name": "filesystem",
        "description": "Read, write, list files in a directory",
        "config": {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-filesystem", "/Users/you/Desktop"]
        }
    },
    {
        "name": "brave-search",
        "description": "Web search via Brave API",
        "config": {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-brave-search"],
            "env": {"BRAVE_API_KEY": "<your-key>"}
        }
    },
    {
        "name": "github",
        "description": "Create issues, PRs, search code on GitHub",
        "config": {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-github"],
            "env": {"GITHUB_PERSONAL_ACCESS_TOKEN": "<your-pat>"}
        }
    },
    {
        "name": "postgres",
        "description": "Query a PostgreSQL database",
        "config": {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-postgres", "postgresql://localhost/mydb"]
        }
    },
    {
        "name": "puppeteer",
        "description": "Headless browser — screenshot, click, scrape pages",
        "config": {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-puppeteer"]
        }
    },
    {
        "name": "google-maps",
        "description": "Directions, geocoding, place search",
        "config": {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-google-maps"],
            "env": {"GOOGLE_MAPS_API_KEY": "<your-key>"}
        }
    },
]

print(f"{'Server':<16} {'Description'}")
print("-" * 65)
for s in reference_servers:
    has_env = "🔑 needs API key" if "env" in s["config"] else "🆓 no key needed"
    print(f"{s['name']:<16} {s['description']:<40} {has_env}")

print("\nExample multi-server config:")
multi = {"mcpServers": {s["name"]: s["config"] for s in reference_servers[:3]}}
print(json.dumps(multi, indent=2))

Server           Description
-----------------------------------------------------------------
filesystem       Read, write, list files in a directory   🆓 no key needed
brave-search     Web search via Brave API                 🔑 needs API key
github           Create issues, PRs, search code on GitHub 🔑 needs API key
postgres         Query a PostgreSQL database              🆓 no key needed
puppeteer        Headless browser — screenshot, click, scrape pages 🆓 no key needed
google-maps      Directions, geocoding, place search      🔑 needs API key

Example multi-server config:
{
  "mcpServers": {
    "filesystem": {
      "command": "npx",
      "args": [
        "-y",
        "@modelcontextprotocol/server-filesystem",
        "/Users/you/Desktop"
      ]
    },
    "brave-search": {
      "command": "npx",
      "args": [
        "-y",
        "@modelcontextprotocol/server-brave-search"
      ],
      "env": {
        "BRAVE_API_KEY": "<your-key>"
      }
    },
    "github": {
      "com

---
## 🔄 End-to-End Workflow — From Code to Claude

Here's the complete development loop you'll follow every time you build or update an MCP server:

```
 1. Write / edit server code
        │
        ▼
 2. Quick sanity check
    python3 server.py          ← should start without crashing
    Ctrl+C to stop
        │
        ▼
 3. Interactive debug with Inspector
    npx @modelcontextprotocol/inspector python3 server.py
    → Open http://localhost:5173
    → Test each tool, check message logs
        │
        ▼
 4. Register in claude_desktop_config.json
    (add mcpServers entry with absolute path)
        │
        ▼
 5. Restart Claude Desktop
    Quit completely → relaunch
        │
        ▼
 6. Verify 🔨 hammer icon appears in chat
    Click → your tools should be listed
        │
        ▼
 7. Chat with Claude — it will call your tools automatically!
    "What is 42 multiplied by 58?"
    "Show me my calculation history"
```

In [10]:
# [PYTHON] ── Complete setup checklist ────────────────────────────────────────

import subprocess, pathlib, sys

def run_check(label: str, test_fn) -> bool:
    try:
        ok = test_fn()
        print(f"  {'✅' if ok else '❌'}  {label}")
        return ok
    except Exception as e:
        print(f"  ❌  {label} — {e}")
        return False

def cmd_version(cmd):
    r = subprocess.run([cmd, "--version"], capture_output=True, timeout=5)
    return r.returncode == 0

print("═" * 50)
print("  Module 4 Setup Checklist")
print("═" * 50)

results = [
    run_check("Python 3.10+",     lambda: sys.version_info >= (3, 10)),
    run_check("pip installed",    lambda: cmd_version("pip")),
    run_check("uv installed",     lambda: cmd_version("uv")),
    run_check("Node.js 18+",      lambda: cmd_version("node")),
    run_check("npm installed",    lambda: cmd_version("npm")),
    run_check("npx available",    lambda: cmd_version("npx")),
    run_check("mcp SDK (Python)", lambda: __import__("mcp") and True),
    run_check("calculator_server.py exists", lambda: pathlib.Path("calculator_server.py").exists()),
]

passed = sum(results)
total  = len(results)
print(f"\n  {passed}/{total} checks passed")
if passed == total:
    print("  🎉 Environment is fully ready!")
else:
    print("  ⚠️  Fix the items marked ❌ above before proceeding to Module 5.")

══════════════════════════════════════════════════
  Module 4 Setup Checklist
══════════════════════════════════════════════════
  ✅  Python 3.10+
  ✅  pip installed
  ✅  uv installed
  ❌  Node.js 18+ — [WinError 2] The system cannot find the file specified
  ❌  npm installed — [WinError 2] The system cannot find the file specified
  ❌  npx available — [WinError 2] The system cannot find the file specified
  ✅  mcp SDK (Python)
  ✅  calculator_server.py exists

  5/8 checks passed
  ⚠️  Fix the items marked ❌ above before proceeding to Module 5.
